Preprocesamiento - Accidentalidad Caldas
Dataset de accidentes de tránsito con columnas: RADICADO, FECHA, HORA, DÍA DE LA SEMANA, CLASE, DIRECCIÓN, GRAVEDAD, BARRIO, DISEÑO.

-Limpieza: detección y tratamiento de valores disfrazados (`SIN INFORMACION`, `DESCONOCIDO`)
-Transformación: FECHA, HORA, variables categóricas

## 1. Importar librerías

In [29]:
import numpy as np                        # Operaciones matemáticas con vectores y matrices
import pandas as pd                       # Manejo de DataFrames (tablas de datos)
import seaborn as sns                     # Visualización estadística
import matplotlib.pyplot as plt           # Gráficos base

from scipy.stats import chi2_contingency  # Prueba Chi-cuadrado (categórica vs categórica)
from scipy.stats import ttest_ind         # Prueba t de Student (numérica vs categórica)

import warnings
warnings.filterwarnings('ignore')

## 2. Cargar el dataset
`nrows=1000` carga solo las primeras 1000 filas para no sobrecargar la memoria durante el análisis exploratorio.

In [30]:
df = pd.read_csv('../raw/accidentalidad_caldas_2003_2025.csv', sep=',', index_col=None)

print('Dimensiones:', df.shape)
display(df.head(10))

Dimensiones: (12635, 1)


,"RADICADO,""FECHA 1"",""HORA"",""DÍA DE LA SEMANA"",""CLASE"",""DIRECCIÓN"",""GRAVEDAD"",""BARRIO"",""DISEÑO"""
0,"2.897,""2004-11-22"",""5:32:00"",""LUNES"",""CHOQUE"",..."
1,"2.898,""2004-11-10"",""6:40:00"",""MIERCOLES"",""CHOQ..."
2,"2.899,""2004-11-29"",""10:10:00"",""LUNES"",""CHOQUE""..."
3,"2.900,""2004-11-27"",""2:45:00"",""SABADO"",""CHOQUE""..."
4,"2.901,""2004-11-28"",""12:01:00"",""DOMINGO"",""ATROP..."
5,"2.902,""2004-11-11"",""3:20:00"",""JUEVES"",""CHOQUE""..."
6,"2.903,""2004-11-22"",""9:40:00"",""LUNES"",""CHOQUE"",..."
7,"2.904,""2004-11-11"",""10:00:00"",""JUEVES"",""VOLCAM..."
8,"2.905,""2004-11-23"",""2:00:00"",""MARTES"",""VOLCAMI..."
9,"2.906,""2004-11-12"",""8:10:00"",""VIERNES"",""CHOQUE..."


3. Información general del dataset
Revisamos tipos de datos y cuántos valores no nulos tiene cada columna.

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12635 entries, 0 to 12634
Data columns (total 1 columns):
 #   Column                                                                                         Non-Null Count  Dtype
---  ------                                                                                         --------------  -----
 0   RADICADO,"FECHA 1","HORA","DÍA DE LA SEMANA","CLASE","DIRECCIÓN","GRAVEDAD","BARRIO","DISEÑO"  12635 non-null  str  
dtypes: str(1)
memory usage: 98.8 KB


4. Limpieza: detección de valores disfrazados
Valores como SIN INFORMACION o DESCONOCIDOvson datos faltantes disfrazadosvde texto.
Estrategia:
- Reemplazarlos por NaN (valor nulo real de Python/Pandas)
- Evaluar cuántos hay por columna


In [32]:
# Lista de valores que consideramos como 'dato faltante disfrazado'
valores_disfrazados = ['SIN INFORMACION', 'DESCONOCIDO']

# Reemplazamos esos valores por NaN en todo el dataframe
# np.nan es el valor estándar de Python para representar un dato faltante
df.replace(valores_disfrazados, np.nan, inplace=True)

# Contamos cuántos nulos hay ahora por columna
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje})
print('Resumen de valores faltantes después del reemplazo:')
display(resumen_nulos)

Resumen de valores faltantes después del reemplazo:


,Nulos,Porcentaje (%)
"RADICADO,""FECHA 1"",""HORA"",""DÍA DE LA SEMANA"",""CLASE"",""DIRECCIÓN"",""GRAVEDAD"",""BARRIO"",""DISEÑO""",0,0.0


4.1 Decisión por columna
- dirección: casi 100% faltante= la eliminamos (no aporta información útil)
- barrio: aproximado de60% faltante= creamos categoría Desconocido (no conviene imputar ni eliminar)
- diseño: aproximado 15% faltante= creamos categoría Desconocido

In [33]:
# DIRECCIÓN tiene demasiados faltantes la eliminamos del dataframe
# axis=1 indica que eliminamos una columna (axis=0 eliminaría filas)
df.drop(columns=['DIRECCIÓN'], inplace=True)
print('Columna DIRECCIÓN eliminada.')

# BARRIO y DISEÑO: rellenamos los NaN con la categoría 'Desconocido'
# Esto preserva la información de que el dato no estaba disponible
df['BARRIO'].fillna('Desconocido', inplace=True)
df['DISEÑO'].fillna('Desconocido', inplace=True)
print('NaN en BARRIO y DISEÑO reemplazados por "Desconocido".')

print()
print('Nulos restantes por columna:')
print(df.isnull().sum())

KeyError: "['DIRECCIÓN'] not found in axis"

##5. Transformación de FECHA
La columna FECHA 1 viene como texto. La convertimos a formato fecha y extraemos:
- AÑO, MES, DIA_MES: para análisis temporal
- ES_FIN_SEMANA`: variable binaria (1 = sábado o domingo, 0 = entre semana)

In [ ]:
# Convertimos la columna de texto a formato fecha real
df['FECHA 1'] = pd.to_datetime(df['FECHA 1'])

# Extraemos componentes de la fecha
df['AÑO']     = df['FECHA 1'].dt.year          # Año (ej: 2019)
df['MES']     = df['FECHA 1'].dt.month         # Mes numérico (1=enero ... 12=diciembre)
df['DIA_MES'] = df['FECHA 1'].dt.day           # Día del mes (1–31)

# dt.dayofweek: 0=lunes ... 6=domingo
# >= 5 significa sábado o domingo → fin de semana
df['ES_FIN_SEMANA'] = (df['FECHA 1'].dt.dayofweek >= 5).astype(int)

print('Nuevas columnas creadas a partir de FECHA:')
display(df[['FECHA 1', 'AÑO', 'MES', 'DIA_MES', 'ES_FIN_SEMANA']].head(8))

##6. Transformación de HORA
La columna HORA viene como texto (`'5:32:00'`). Aplicamos dos estrategias:
- Opción 1: extraer la hora como número entero (0–23) → útil para correlaciones
- Opción 2 (Discretización): convertir en franjas horarias → más interpretable

> Discretización es dividir una variable continua en categorías con significado.

In [ ]:
# Extraemos solo la parte de la hora como número entero
# .str.split(':') divide '5:32:00' en ['5','32','00']
# .str[0] toma el primer elemento ('5')
df['HORA_NUM'] = df['HORA'].str.split(':').str[0].astype(float)

# Opción 2: Discretización en franjas horarias
# pd.cut divide un rango numérico en intervalos que nosotros definimos
bins   = [-1, 5, 11, 17, 23]                                   # Límites de los intervalos
labels = ['Madrugada (0-5)', 'Mañana (6-11)',
          'Tarde (12-17)',   'Noche (18-23)']                   # Nombre de cada franja

df['FRANJA_HORARIA'] = pd.cut(df['HORA_NUM'], bins=bins, labels=labels)

print('Distribución por franja horaria:')
print(df['FRANJA_HORARIA'].value_counts())
print()
display(df[['HORA', 'HORA_NUM', 'FRANJA_HORARIA']].head(8))

##7. Codificación de variables categóricas. Aplicamos:
- Label Encoding para GRAVEDAD y DÍA DE LA SEMANA (tienen orden lógico)
- One Hot Encoding para CLASE y DISEÑO (sin orden, varias categorías)

In [ ]:
df_enc = df.copy()  # Trabajamos sobre una copia para preservar el original

# --- Label Encoding: GRAVEDAD ---
# Asignamos un número según severidad creciente
mapa_gravedad = {'DAÑOS': 0, 'HERIDOS': 1, 'MUERTOS': 2}
df_enc['GRAVEDAD_cod'] = df_enc['GRAVEDAD'].map(mapa_gravedad)

# --- Label Encoding: DÍA DE LA SEMANA ---
# Asignamos un número del 0 (lunes) al 6 (domingo) para preservar el orden semanal
orden_dias = ['LUNES','MARTES','MIERCOLES','JUEVES','VIERNES','SABADO','DOMINGO']
df_enc['DIA_cod'] = pd.Categorical(
    df_enc['DÍA DE LA SEMANA'], categories=orden_dias, ordered=True
).codes

# --- One Hot Encoding: CLASE y DISEÑO ---
# Crea una columna binaria (0 o 1) por cada categoría
# drop_first=True evita multicolinealidad (elimina una columna redundante)
df_enc = pd.get_dummies(df_enc, columns=['CLASE', 'DISEÑO'], drop_first=False)

print('Columnas después de codificación:')
print(df_enc.columns.tolist())

## 8. Dataset final preprocesado
Mostramos el dataframe con todas las transformaciones aplicadas

In [ ]:
cols_finales = ['RADICADO', 'FECHA 1', 'AÑO', 'MES', 'DIA_MES', 'ES_FIN_SEMANA',
                'HORA_NUM', 'FRANJA_HORARIA', 'DÍA DE LA SEMANA',
                'CLASE', 'GRAVEDAD', 'BARRIO', 'DISEÑO']

# DIA_cod y GRAVEDAD_cod están en df_enc (la copia codificada)
# Los unimos al dataframe principal para mostrarlos juntos
df['DIA_cod']      = df_enc['DIA_cod']
df['GRAVEDAD_cod'] = df_enc['GRAVEDAD_cod']

cols_finales = cols_finales + ['DIA_cod', 'GRAVEDAD_cod']

print('Dataset final preprocesado:')
display(df[cols_finales].head(10))
print('Dimensiones:', df[cols_finales].shape)

## 9. Guardar dataset limpio

In [ ]:
df[cols_finales].to_csv('../datasets_limpios/accidentalidad_caldas_limpio.csv', sep=';', index=False, encoding='utf-8')
print('Dataset guardado correctamente en: ../datasets_limpios/accidentalidad_caldas_limpio.csv')